# Discovery y Perfilado de Datos

In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine

In [2]:
user = os.environ.get("WAREHOUSE_DB_USER", "postgres")
password = os.environ.get("WAREHOUSE_DB_PASSWORD", "postgres")
db = os.environ.get("WAREHOUSE_DB_NAME", "warehouse")
host = os.environ.get("WAREHOUSE_DB_HOST", "warehouse-db")
port = os.environ.get("WAREHOUSE_DB_PORT", "5432")

In [3]:
engine = create_engine(f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{db}")

### Tabla semester

In [10]:
#Nulos
query = """
SELECT
  COUNT(*) AS total,
  COUNT(*) FILTER (WHERE semester_id IS NULL) AS semester_id,
  COUNT(*) FILTER (WHERE code IS NULL) AS code,
  COUNT(*) FILTER (WHERE year IS NULL) AS year,
  COUNT(*) FILTER (WHERE half IS NULL) AS half,
  COUNT(*) FILTER (WHERE start_date IS NULL) AS start_date,
  COUNT(*) FILTER (WHERE end_date IS NULL) AS end_date
FROM bronze.university_semesters;"""
df = pd.read_sql(query, engine)
df

,total,semester_id,code,year,half,start_date,end_date
0,8,0,0,0,0,0,0


In [14]:
#Duplicados
query = """
SELECT code, COUNT(*) AS n
    FROM bronze.university_semesters
    GROUP BY code
    HAVING COUNT(*) > 1;
"""
df = pd.read_sql(query, engine)
df

,code,n


In [29]:
#Consistencia code contra year - half
query = """
    SELECT semester_id, code, year, half,
      CASE
        WHEN half = 1 THEN '1-' || year::text
        WHEN half = 2 THEN '2-' || year::text
        ELSE NULL
      END AS code_esperado
    FROM bronze.university_semesters
    WHERE code IS DISTINCT FROM (
      CASE
        WHEN half = 1 THEN '1-' || year::text 
        WHEN half = 2 THEN '2-' || year::text 
        ELSE NULL
      END
    );
"""
df = pd.read_sql(query, engine)
df

,semester_id,code,year,half,code_esperado
0,SEM-001,2022-1,2022,1,1-2022
1,SEM-002,2022-2,2022,2,2-2022
2,SEM-003,2023-1,2023,1,1-2023
3,SEM-004,2023-2,2023,2,2-2023
4,SEM-005,2024-1,2024,1,1-2024
5,SEM-006,2024-2,2024,2,2-2024
6,SEM-007,2025-1,2025,1,1-2025
7,SEM-008,2025-2,2025,2,2-2025


In [32]:
#start_date debe ser menor que end_date
query = """
    SELECT *
    FROM bronze.university_semesters
    WHERE start_date >= end_date;
"""
df = pd.read_sql(query, engine)
df

,semester_id,code,year,half,start_date,end_date


In [33]:
#Semestres solapados
query = """
    SELECT a.semester_id AS semestre_a, b.semester_id AS semestre_b,
           a.start_date AS inicio_a, a.end_date AS fin_a,
           b.start_date AS inicio_b, b.end_date AS fin_b
    FROM bronze.university_semesters a
    JOIN bronze.university_semesters b
      ON a.semester_id < b.semester_id
     AND a.start_date <= b.end_date
     AND b.start_date <= a.end_date;
"""
df = pd.read_sql(query, engine)
df

,semestre_a,semestre_b,inicio_a,fin_a,inicio_b,fin_b


In [35]:
#Distribución de duración del semestre (días)
query = """
    SELECT 
        semester_id, 
        code, 
        (CAST(end_date AS DATE) - CAST(start_date AS DATE)) AS duracion_dias
    FROM bronze.university_semesters
    ORDER BY duracion_dias;
"""
df = pd.read_sql(query, engine)
df

,semester_id,code,duracion_dias
0,SEM-001,2022-1,136
1,SEM-002,2022-2,136
2,SEM-003,2023-1,136
3,SEM-004,2023-2,136
4,SEM-005,2024-1,136
5,SEM-006,2024-2,136
6,SEM-007,2025-1,136
7,SEM-008,2025-2,136


In [37]:
#Cardinalidad de half
query = """
    SELECT half, COUNT(*) AS n
    FROM bronze.university_semesters
    GROUP BY half;
"""
df = pd.read_sql(query, engine)
df

,half,n
0,2,4
1,1,4


In [ ]:
query = """
    
    FROM bronze.university_;
"""
df = pd.read_sql(query, engine)
df

### Tabla professors

In [ ]:
#Exploración general
query = """
    SELECT
      COUNT(*) AS total,
      COUNT(DISTINCT professor_id) AS ids,
      COUNT(*) FILTER (WHERE first_name IS NULL) AS first_name,
      COUNT(*) FILTER (WHERE last_name IS NULL) AS last_name,
      COUNT(*) FILTER (WHERE email IS NULL) AS email,
      COUNT(*) FILTER (WHERE department IS NULL) AS department,
      COUNT(*) FILTER (WHERE hired_at IS NULL) AS hired_at
    FROM bronze.university_professors;
"""
df = pd.read_sql(query, engine)
df

,total,ids,first_name,last_name,email,department,hired_at
0,200,200,0,0,0,0,0


In [74]:
#Posibles profesores duplicadas
query = """    
    SELECT first_name, last_name, COUNT(*) AS n
    FROM bronze.university_professors
    GROUP BY first_name, last_name
    HAVING COUNT(*) > 1;
"""
df = pd.read_sql(query, engine)
df

,first_name,last_name,n
0,Isabel,Pino,2
1,Valentina,Fuentes,2
2,Matias,Romero,2
3,Martina,Diaz,2
4,Benjamin,Munoz,2
5,Amanda,Pino,2


In [41]:
#Emails duplicados
query = """
    SELECT email, COUNT(*) AS n
    FROM bronze.university_professors
    GROUP BY email
    HAVING COUNT(*) > 1;
"""
df = pd.read_sql(query, engine)
df

,email,n


In [42]:
#Distribución de department
query = """
    SELECT department, COUNT(*) AS n_profesores
    FROM bronze.university_professors
    GROUP BY department
    ORDER BY n_profesores DESC;
"""
df = pd.read_sql(query, engine)
df

,department,n_profesores
0,cs,53
1,economics,31
2,math,31
3,history,20
4,physics,17
5,literature,17
6,biology,17
7,chemistry,14


In [45]:
#Rango de hired_at e histograma anual
query = """
    SELECT MIN(hired_at) AS primera_contratacion, MAX(hired_at) AS ultima_contratacion
    FROM bronze.university_professors;
"""
df = pd.read_sql(query, engine)
df

,primera_contratacion,ultima_contratacion
0,2005-03-10,2024-12-27


In [47]:
#Rango de hired_at e histograma anual
query = """   
    SELECT 
    EXTRACT(YEAR FROM hired_at::date) AS year, 
    COUNT(*) AS n_contrataciones
    FROM bronze.university_professors
    GROUP BY 1
    ORDER BY 1;
"""
df = pd.read_sql(query, engine)
df

,year,n_contrataciones
0,2005.0,8
1,2006.0,14
2,2007.0,12
3,2008.0,9
4,2009.0,9
5,2010.0,5
6,2011.0,12
7,2012.0,8
8,2013.0,11
9,2014.0,9


In [49]:
#Profesores que dictan cursos de otro department
query = """
    SELECT p.professor_id, p.department AS depto_profesor,
       c.course_id, c.department AS depto_curso
    FROM bronze.university_professors p
    JOIN bronze.university_courses c ON c.professor_id = p.professor_id
    WHERE p.department <> c.department; 
"""
df = pd.read_sql(query, engine)
df

,professor_id,depto_profesor,course_id,depto_curso
0,PRF-00153,history,CRS-00001,cs
1,PRF-00025,chemistry,CRS-00003,literature
2,PRF-00163,cs,CRS-00004,physics
3,PRF-00130,cs,CRS-00005,biology
4,PRF-00059,biology,CRS-00006,history
...,...,...,...,...
259,PRF-00109,cs,CRS-00296,chemistry
260,PRF-00154,math,CRS-00297,cs
261,PRF-00024,math,CRS-00298,history
262,PRF-00026,cs,CRS-00299,biology


In [53]:
#Cursos por profesor (carga docente)
query = """
    SELECT p.professor_id, p.first_name, p.last_name, COUNT(c.course_id) AS n_cursos
    FROM bronze.university_professors p
    LEFT JOIN bronze.university_courses c ON c.professor_id = p.professor_id
    GROUP BY p.professor_id, p.first_name, p.last_name
    ORDER BY n_cursos DESC;
"""
df = pd.read_sql(query, engine)
df

,professor_id,first_name,last_name,n_cursos
0,PRF-00051,Josefa,Sepulveda,6
1,PRF-00162,Sofia,Valenzuela,5
2,PRF-00130,Diego,Munoz,4
3,PRF-00099,Sebastian,Gomez,4
4,PRF-00127,Benjamin,Munoz,4
...,...,...,...,...
195,PRF-00174,Juan,Martinez,0
196,PRF-00009,Vicente,Rivera,0
197,PRF-00046,Paula,Carvajal,0
198,PRF-00185,Magdalena,Pizarro,0


### Tabla students

In [55]:
query = """
    SELECT
      COUNT(*) AS total,
      COUNT(DISTINCT student_id) AS ids,
      COUNT(*) FILTER (WHERE email IS NULL) AS email,
      COUNT(*) FILTER (WHERE birth_date IS NULL) AS birth_date,
      COUNT(*) FILTER (WHERE enrolled_at IS NULL) AS enrolled_at,
      COUNT(*) FILTER (WHERE country IS NULL) AS country 
    FROM bronze.university_students;
"""
df = pd.read_sql(query, engine)
df

,total,ids,email,birth_date,enrolled_at,country
0,5000,5000,0,0,0,0


In [58]:
#Emails duplicados
query = """
    SELECT email, COUNT(*) AS n
    FROM bronze.university_students
    GROUP BY email
    HAVING COUNT(*) > 1;
"""
df = pd.read_sql(query, engine)
df

,email,n


In [67]:
#Posibles estudiantes duplicados
query = """    
    SELECT first_name, last_name, birth_date, COUNT(*) AS n
    FROM bronze.university_students
    GROUP BY first_name, last_name, birth_date
    HAVING COUNT(*) > 1;
"""
df = pd.read_sql(query, engine)
df

,first_name,last_name,birth_date,n
0,Diego,Carvajal,1999-08-23,2
1,Maria,Vega,1997-10-20,2
2,Cristobal,Torres,2007-11-05,2


In [68]:
#Distribución de country
query = """    
    SELECT country, COUNT(*) AS n_estudiantes
    FROM bronze.university_students
    GROUP BY country
    ORDER BY n_estudiantes DESC;
"""
df = pd.read_sql(query, engine)
df

,country,n_estudiantes
0,CL,1980
1,AR,523
2,PE,513
3,MX,483
4,BR,427
5,ES,390
6,CO,389
7,US,295


In [79]:
#Distribución de edad
query = """    
    SELECT 
        MIN(EXTRACT(YEAR FROM age(birth_date::date))) AS edad_minima,
        MAX(EXTRACT(YEAR FROM age(birth_date::date))) AS edad_maxima
    FROM bronze.university_students;
"""
df = pd.read_sql(query, engine)
df

,edad_minima,edad_maxima
0,18.0,31.0


In [83]:
#Distribución de edad
query = """    
    SELECT 
        CASE 
            WHEN date_part('year', age(birth_date::date)) < 18 THEN 'Menor de 18'
            WHEN date_part('year', age(birth_date::date)) BETWEEN 18 AND 21 THEN '18 - 21'
            WHEN date_part('year', age(birth_date::date)) BETWEEN 22 AND 25 THEN '22 - 25'
            WHEN date_part('year', age(birth_date::date)) BETWEEN 26 AND 30 THEN '26 - 30'
            ELSE '31+'
        END AS rango_edad,
        COUNT(*) AS n
    FROM bronze.university_students
    GROUP BY 1
    ORDER BY 1;
"""
df = pd.read_sql(query, engine)
df

,rango_edad,n
0,18 - 21,1331
1,22 - 25,1547
2,26 - 30,1912
3,31+,210


In [86]:
#enrolled_at y primera inscripción real
query = """    
    SELECT MIN(enrolled_at) AS primer_ingreso, MAX(enrolled_at) AS ultimo_ingreso
    FROM bronze.university_students
"""
df = pd.read_sql(query, engine)
df

,primer_ingreso,ultimo_ingreso
0,2018-01-01,2025-09-30


In [87]:
#enrolled_at y primera inscripción real
query = """        
    SELECT s.student_id, s.enrolled_at AS fecha_alta_estudiante,
           MIN(e.enrolled_at) AS primera_inscripcion_curso
    FROM bronze.university_students s
    JOIN bronze.university_enrollments e ON e.student_id = s.student_id
    GROUP BY s.student_id, s.enrolled_at
    HAVING s.enrolled_at > MIN(e.enrolled_at);
"""
df = pd.read_sql(query, engine)
df

,student_id,fecha_alta_estudiante,primera_inscripcion_curso
0,STU-0000003,2022-10-19,2022-05-20
1,STU-0000795,2024-05-03,2022-05-30
2,STU-0003096,2024-05-29,2022-07-24
3,STU-0001339,2022-11-08,2022-02-08
4,STU-0004181,2023-11-30,2022-06-12
...,...,...,...
1846,STU-0003410,2023-05-25,2022-11-07
1847,STU-0001832,2025-05-26,2023-04-25
1848,STU-0004551,2023-01-28,2022-02-19
1849,STU-0004409,2022-04-28,2022-04-07


In [90]:
#Estudiantes sin ninguna inscripción (huérfanos)
query = """
    SELECT s.student_id
    FROM bronze.university_students s
    LEFT JOIN bronze.university_enrollments e ON e.student_id = s.student_id
    WHERE e.enrollment_id IS NULL;
"""
df = pd.read_sql(query, engine)
df

,student_id
0,STU-0002862
1,STU-0003834
2,STU-0002858
3,STU-0003929
4,STU-0000958
5,STU-0001529
6,STU-0002341
7,STU-0001864
8,STU-0000485
9,STU-0002571


### Tabla courses

In [93]:
# Perfil general
query = """
    SELECT
      COUNT(*) AS total,
      COUNT(DISTINCT course_id) AS ids,
      COUNT(DISTINCT code) AS codes,
      COUNT(*) FILTER (WHERE credits IS NULL) AS credits,
      COUNT(*) FILTER (WHERE department IS NULL) AS department,
      COUNT(*) FILTER (WHERE professor_id IS NULL) AS professor_id
    FROM bronze.university_courses;
"""
df = pd.read_sql(query, engine)
df

,total,ids,codes,credits,department,professor_id
0,300,300,300,0,0,0


In [94]:
#Distribución de credits
query = """
    SELECT MIN(credits) AS min_creditos, MAX(credits) AS max_creditos, AVG(credits) AS promedio_creditos
    FROM bronze.university_courses;
"""
df = pd.read_sql(query, engine)
df

,min_creditos,max_creditos,promedio_creditos
0,2,6,3.98


In [95]:
query = """
    SELECT credits, COUNT(*) AS n_cursos
    FROM bronze.university_courses
    GROUP BY credits
    ORDER BY credits;
"""
df = pd.read_sql(query, engine)
df

,credits,n_cursos
0,2,64
1,3,59
2,4,58
3,5,57
4,6,62


In [96]:
#Distribución de department
query = """
    SELECT department, COUNT(*) AS n_cursos
    FROM bronze.university_courses
    GROUP BY department
    ORDER BY n_cursos DESC;
"""
df = pd.read_sql(query, engine)
df

,department,n_cursos
0,physics,52
1,cs,47
2,biology,46
3,economics,35
4,literature,34
5,chemistry,32
6,history,31
7,math,23


In [97]:
#Inscripciones por curso
query = """
    SELECT c.course_id, c.code, c.name, COUNT(e.enrollment_id) AS n_inscripciones
    FROM bronze.university_courses c
    LEFT JOIN bronze.university_enrollments e ON e.course_id = c.course_id
    GROUP BY c.course_id, c.code, c.name
    ORDER BY n_inscripciones DESC; 
"""
df = pd.read_sql(query, engine)
df

,course_id,code,name,n_inscripciones
0,CRS-00190,C-00190,Course 00190,108
1,CRS-00053,C-00053,Course 00053,108
2,CRS-00042,C-00042,Course 00042,105
3,CRS-00245,C-00245,Course 00245,104
4,CRS-00010,C-00010,Course 00010,104
...,...,...,...,...
295,CRS-00283,C-00283,Course 00283,66
296,CRS-00234,C-00234,Course 00234,66
297,CRS-00024,C-00024,Course 00024,66
298,CRS-00225,C-00225,Course 00225,64


In [100]:
#Cursos sin profesor
query = """
    SELECT *
    FROM bronze.university_courses
    WHERE professor_id IS NULL;
"""
df = pd.read_sql(query, engine)
df

,course_id,code,name,credits,department,professor_id


In [103]:
#Códigos de curso duplicados
query = """
    SELECT code, COUNT(*) AS n
    FROM bronze.university_courses
    GROUP BY code
    HAVING COUNT(*) > 1;
"""
df = pd.read_sql(query, engine)
df

,code,n


### Tabla enrollments

In [104]:
#Perfil general
query = """
SELECT
  COUNT(*) AS total,
  COUNT(DISTINCT enrollment_id) AS ids,
  COUNT(*) FILTER (WHERE enrolled_at IS NULL) AS enrolled_at,
  COUNT(*) FILTER (WHERE status IS NULL) AS status,
  COUNT(*) FILTER (WHERE student_id IS NULL) AS student_id,
  COUNT(*) FILTER (WHERE course_id IS NULL) AS course_id,
  COUNT(*) FILTER (WHERE semester_id IS NULL) AS semester_id
FROM bronze.university_enrollments;
"""
df = pd.read_sql(query, engine)
df

,total,ids,enrolled_at,status,student_id,course_id,semester_id
0,25000,25000,0,0,0,0,0


In [105]:
#Distribución de status
query = """
SELECT status, COUNT(*) AS n
FROM bronze.university_enrollments
GROUP BY status
ORDER BY n DESC;
"""
df = pd.read_sql(query, engine)
df

,status,n
0,completed,14931
1,active,5035
2,failed,2531
3,dropped,2503


In [106]:
#Duplicados por clave de negocio (student+course+semester)
query = """
SELECT student_id, course_id, semester_id, COUNT(*) AS n
FROM bronze.university_enrollments
GROUP BY student_id, course_id, semester_id
HAVING COUNT(*) > 1;
"""
df = pd.read_sql(query, engine)
df

,student_id,course_id,semester_id,n
0,STU-0002362,CRS-00282,SEM-002,2
1,STU-0004433,CRS-00114,SEM-002,2
2,STU-0002200,CRS-00049,SEM-004,2
3,STU-0003756,CRS-00148,SEM-004,2
4,STU-0001575,CRS-00112,SEM-007,2
5,STU-0003848,CRS-00296,SEM-006,2
6,STU-0001847,CRS-00043,SEM-003,2
7,STU-0002714,CRS-00047,SEM-005,2
8,STU-0000994,CRS-00116,SEM-004,2
9,STU-0003979,CRS-00274,SEM-004,2


In [107]:
#Inscripciones por estudiante (distribución)
query = """
SELECT student_id, COUNT(*) AS n_inscripciones
FROM bronze.university_enrollments
GROUP BY student_id
ORDER BY n_inscripciones DESC;
"""
df = pd.read_sql(query, engine)
df

,student_id,n_inscripciones
0,STU-0003693,16
1,STU-0004367,14
2,STU-0002482,13
3,STU-0004696,13
4,STU-0001754,13
...,...,...
4957,STU-0003023,1
4958,STU-0000617,1
4959,STU-0001964,1
4960,STU-0001162,1


In [108]:
query = """
SELECT n_inscripciones, COUNT(*) AS n_estudiantes
FROM (
  SELECT student_id, COUNT(*) AS n_inscripciones
  FROM bronze.university_enrollments
  GROUP BY student_id
) t
GROUP BY n_inscripciones
ORDER BY n_inscripciones;
"""
df = pd.read_sql(query, engine)
df

,n_inscripciones,n_estudiantes
0,1,158
1,2,403
2,3,713
3,4,874
4,5,902
5,6,749
6,7,501
7,8,326
8,9,182
9,10,93


In [112]:
#Inscripciones por semestre
query = """
SELECT semester_id, COUNT(*) AS n_inscripciones
FROM bronze.university_enrollments
GROUP BY semester_id
ORDER BY semester_id;
"""
df = pd.read_sql(query, engine)
df

,semester_id,n_inscripciones
0,SEM-001,3164
1,SEM-002,3199
2,SEM-003,3143
3,SEM-004,3014
4,SEM-005,3076
5,SEM-006,3108
6,SEM-007,3110
7,SEM-008,3186


In [113]:
#enrolled_at fuera del rango del semestre
query = """
SELECT e.enrollment_id, e.enrolled_at, s.semester_id, s.start_date, s.end_date
FROM bronze.university_enrollments e
JOIN bronze.university_semesters s ON s.semester_id = e.semester_id
WHERE e.enrolled_at < s.start_date OR e.enrolled_at > s.end_date;
"""
df = pd.read_sql(query, engine)
df

,enrollment_id,enrolled_at,semester_id,start_date,end_date
0,ENR-00000001,2022-03-25,SEM-004,2023-08-01,2023-12-15
1,ENR-00000002,2025-03-05,SEM-003,2023-03-01,2023-07-15
2,ENR-00000003,2024-07-12,SEM-002,2022-08-01,2022-12-15
3,ENR-00000004,2023-06-21,SEM-004,2023-08-01,2023-12-15
4,ENR-00000005,2023-07-24,SEM-006,2024-08-01,2024-12-15
...,...,...,...,...,...
22724,ENR-00024995,2022-03-20,SEM-008,2025-08-01,2025-12-15
22725,ENR-00024996,2023-02-09,SEM-003,2023-03-01,2023-07-15
22726,ENR-00024997,2022-01-15,SEM-002,2022-08-01,2022-12-15
22727,ENR-00024998,2025-07-03,SEM-006,2024-08-01,2024-12-15


In [114]:
#Notas calificadas antes de la inscripción
query = """
SELECT e.enrollment_id, e.enrolled_at, g.grade_id, g.graded_at
FROM bronze.university_enrollments e
JOIN bronze.university_grades g ON g.enrollment_id = e.enrollment_id
WHERE g.graded_at < e.enrolled_at;
"""
df = pd.read_sql(query, engine)
df

,enrollment_id,enrolled_at,grade_id,graded_at
0,ENR-00002100,2024-11-24,GRD-00000001,2024-10-15
1,ENR-00019216,2024-08-21,GRD-00000005,2024-07-11
2,ENR-00014503,2024-04-08,GRD-00000006,2023-03-17
3,ENR-00003147,2025-02-10,GRD-00000007,2023-08-04
4,ENR-00010957,2023-09-05,GRD-00000008,2023-05-29
...,...,...,...,...
29236,ENR-00023776,2025-10-21,GRD-00059991,2023-07-01
29237,ENR-00000186,2025-11-19,GRD-00059994,2024-05-09
29238,ENR-00019934,2025-04-17,GRD-00059998,2024-05-14
29239,ENR-00000988,2024-12-01,GRD-00059999,2022-11-13


In [116]:
#Integridad referencial (FKs huérfanas)
query = """
SELECT e.enrollment_id
FROM bronze.university_enrollments e
LEFT JOIN bronze.university_students s ON s.student_id = e.student_id
WHERE s.student_id IS NULL;
"""
df = pd.read_sql(query, engine)
df

,enrollment_id


In [118]:
query = """
SELECT e.enrollment_id
FROM bronze.university_enrollments e
LEFT JOIN bronze.university_courses c ON c.course_id = e.course_id
WHERE c.course_id IS NULL;
"""
df = pd.read_sql(query, engine)
df

,enrollment_id


In [119]:
query = """
SELECT e.enrollment_id
FROM bronze.university_enrollments e
LEFT JOIN bronze.university_semesters sem ON sem.semester_id = e.semester_id
WHERE sem.semester_id IS NULL;
"""
df = pd.read_sql(query, engine)
df

,enrollment_id


### Tabla grades

In [120]:
# Perfil general
query = """
SELECT
  COUNT(*) AS total,
  COUNT(DISTINCT grade_id) AS ids,
  COUNT(*) FILTER (WHERE assessment IS NULL) AS assessment,
  COUNT(*) FILTER (WHERE score IS NULL) AS score,
  COUNT(*) FILTER (WHERE weight IS NULL) AS weight,
  COUNT(*) FILTER (WHERE graded_at IS NULL) AS graded_at,
  COUNT(*) FILTER (WHERE enrollment_id IS NULL) AS enrollment_id
FROM bronze.university_grades;
"""
df = pd.read_sql(query, engine)
df

,total,ids,assessment,score,weight,graded_at,enrollment_id
0,60000,60000,0,0,0,0,0


In [121]:
#Tipos de assessment (cardinalidad)
query = """
SELECT assessment, COUNT(*) AS n
FROM bronze.university_grades
GROUP BY assessment
ORDER BY n DESC;
"""
df = pd.read_sql(query, engine)
df 

,assessment,n
0,quiz,12132
1,project,12003
2,midterm,11970
3,homework,11966
4,final,11929


In [122]:
#Score: stats general y por assessment
query = """
SELECT MIN(score) AS min_score, MAX(score) AS max_score, AVG(score) AS promedio_score
FROM bronze.university_grades;
"""
df = pd.read_sql(query, engine)
df

,min_score,max_score,promedio_score
0,24.53,100.0,74.884157


In [123]:
query = """
SELECT assessment,
       MIN(score) AS min_score, MAX(score) AS max_score, AVG(score) AS promedio_score,
       COUNT(*) AS n
FROM bronze.university_grades
GROUP BY assessment
ORDER BY assessment;
"""
df = pd.read_sql(query, engine)
df

,assessment,min_score,max_score,promedio_score,n
0,final,27.50,100.0,74.757965,11929
1,homework,24.53,100.0,74.879357,11966
2,midterm,28.92,100.0,74.875388,11970
3,project,28.00,100.0,74.790801,12003
4,quiz,28.54,100.0,75.113986,12132


In [124]:
#Rango de graded_at
query = """
SELECT MIN(graded_at) AS calificacion_mas_antigua, MAX(graded_at) AS calificacion_mas_reciente
FROM bronze.university_grades;
"""
df = pd.read_sql(query, engine)
df

,calificacion_mas_antigua,calificacion_mas_reciente
0,2022-02-01,2025-12-31


In [125]:
#Suma de weight por enrollment (debe sumar ~1 o ~100)
query = """
SELECT enrollment_id, SUM(weight) AS suma_pesos
FROM bronze.university_grades
GROUP BY enrollment_id
HAVING SUM(weight) NOT BETWEEN 0.98 AND 1.02
   AND SUM(weight) NOT BETWEEN 98 AND 102;
"""
df = pd.read_sql(query, engine)
df

,enrollment_id,suma_pesos
0,ENR-00022689,0.71
1,ENR-00017562,1.56
2,ENR-00014785,1.59
3,ENR-00019158,0.27
4,ENR-00015054,0.62
...,...,...
22099,ENR-00016699,0.39
22100,ENR-00007854,0.20
22101,ENR-00000556,0.27
22102,ENR-00013430,0.59


In [126]:
#Duplicados (mismo enrollment_id + assessment)
query = """
SELECT enrollment_id, assessment, COUNT(*) AS n
FROM bronze.university_grades
GROUP BY enrollment_id, assessment
HAVING COUNT(*) > 1; 
"""
df = pd.read_sql(query, engine)
df

,enrollment_id,assessment,n
0,ENR-00024773,project,2
1,ENR-00006280,project,2
2,ENR-00022705,quiz,2
3,ENR-00006046,quiz,2
4,ENR-00008144,midterm,2
...,...,...,...
10539,ENR-00009267,project,2
10540,ENR-00003303,final,2
10541,ENR-00009798,quiz,2
10542,ENR-00010035,homework,2


In [127]:
#Integridad referencial hacia enrollments
query = """
SELECT g.grade_id
FROM bronze.university_grades g
LEFT JOIN bronze.university_enrollments e ON e.enrollment_id = g.enrollment_id
WHERE e.enrollment_id IS NULL;
"""
df = pd.read_sql(query, engine)
df

,grade_id


In [128]:
#graded_at posterior al end_date del semestre correspondiente
query = """
    SELECT g.grade_id, g.graded_at, sem.semester_id, sem.end_date
    FROM bronze.university_grades g
    JOIN bronze.university_enrollments e ON e.enrollment_id = g.enrollment_id
    JOIN bronze.university_semesters sem ON sem.semester_id = e.semester_id
    WHERE g.graded_at > sem.end_date;
"""
df = pd.read_sql(query, engine)
df

,grade_id,graded_at,semester_id,end_date
0,GRD-00000001,2024-10-15,SEM-001,2022-07-15
1,GRD-00000002,2025-05-20,SEM-005,2024-07-15
2,GRD-00000003,2025-01-09,SEM-003,2023-07-15
3,GRD-00000004,2024-12-27,SEM-006,2024-12-15
4,GRD-00000005,2024-07-11,SEM-003,2023-07-15
...,...,...,...,...
26823,GRD-00059987,2024-08-27,SEM-004,2023-12-15
26824,GRD-00059990,2024-12-24,SEM-006,2024-12-15
26825,GRD-00059993,2025-12-09,SEM-001,2022-07-15
26826,GRD-00059995,2024-04-18,SEM-003,2023-07-15


In [129]:
#Score fuera de rango esperado (ajustar límites según negocio)
query = """
    SELECT *
    FROM bronze.university_grades
    WHERE score < 0 OR score > 100;
"""
df = pd.read_sql(query, engine)
df

,grade_id,assessment,score,weight,graded_at,enrollment_id
